# Ablation: Quantization (4-bit vs 8-bit)

Compares quantization impact on quality and speed across both languages.

**Memory strategy**: SigExt on CPU → unloaded → LLM swapped per quantization.

In [ ]:
!pip install -e ../..
from huggingface_hub import login
login()

In [ ]:
from sm_sip.config import SigExtConfig, InferenceConfig
from sm_sip.data import get_test_data
from sm_sip.models import load_sigext_model, unload_sigext_model, load_llm, create_summary_chain, preprocess_dataset
from sm_sip.prompts import get_summary_prompt
from sm_sip.pipelines import run_inference, run_evaluation
from sm_sip.utils.io import save_results
from sm_sip.utils.gpu import clear_gpu_memory
import time

In [ ]:
results = {}
for lang in ['it', 'en']:
    preset = '10k-60t' if lang == 'it' else '1k-60t'
    sc = SigExtConfig.from_preset(lang, preset)
    data = get_test_data(lang=lang, num_samples=50, skip_samples=sc.skip_samples)
    sm, st = load_sigext_model(sc.model_id, device='cpu')
    proc = preprocess_dataset(data, sm, st, lang=lang)
    unload_sigext_model(sm, st)
    for quant in ['4bit', '8bit']:
        key = f'{lang}_{quant}'
        t0 = time.time()
        _, _, pipe = load_llm('meta-llama/Llama-3.1-8B-Instruct', quant)
        chain = create_summary_chain(pipe, get_summary_prompt(lang, 'source_aware'))
        res = run_inference(proc, chain)
        metrics = run_evaluation(res, lang=lang)
        metrics['time_s'] = time.time() - t0
        results[key] = metrics
        clear_gpu_memory()
save_results({'ablation': 'quantization', 'results': results}, 'results/ablation_quantization.json')
print('Done!')